In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. 複雑なテスト関数の定義
def complex_test_function(x, y):
    peak1 = 1.5 / ((x - 1.0)**2 + (y - 1.5)**2 + 0.005)
    peak2 = 1.0 / ((x - 3.0)**2 + (y - 3.5)**2 + 0.05)
    background = 0.5 * np.sin(x) * np.cos(y)
    return peak1 + peak2 + background

# 散乱データの生成 (Scattered Data) - 計251点
np.random.seed(100)
torch.manual_seed(100)
N_train = 250       
x_train = np.random.uniform(0, 4, N_train)
y_train = np.random.uniform(1, 5, N_train)
x_train = np.append(x_train, 1.0)
y_train = np.append(y_train, 1.5)
z_train = complex_test_function(x_train, y_train)

X_train_raw = np.column_stack([x_train, y_train])
X_train_tensor = torch.tensor(X_train_raw, dtype=torch.float32)
Z_train_tensor = torch.tensor(z_train, dtype=torch.float32).view(-1, 1)


# =====================================================================
# 2. 【提案手法】学習可能な有理関数活性化関数の定義
# =====================================================================
class RationalActivation(nn.Module):
    def __init__(self):
        super(RationalActivation, self).__init__()
        # 分子の係数 (P(x) = a0 + a1*x + a2*x^2 + a3*x^3)
        self.a0 = nn.Parameter(torch.randn(1) * 0.1)
        self.a1 = nn.Parameter(torch.randn(1) * 0.1)
        self.a2 = nn.Parameter(torch.randn(1) * 0.1)
        self.a3 = nn.Parameter(torch.randn(1) * 0.1)
        
        # 分母の係数 (Q(x) = 1 + |b1*x + b2*x^2 + b3*x^3|)
        self.b1 = nn.Parameter(torch.randn(1) * 0.1)
        self.b2 = nn.Parameter(torch.randn(1) * 0.1)
        self.b3 = nn.Parameter(torch.randn(1) * 0.1)

    def forward(self, x):
        # P(x) の計算
        P = self.a0 + self.a1 * x + self.a2 * (x**2) + self.a3 * (x**3)
        # Q(x) の計算 (絶対値をつけることで分母がゼロになるのを徹底的に防ぐ)
        Q = 1.0 + torch.abs(self.b1 * x + self.b2 * (x**2) + self.b3 * (x**3))
        return P / Q


# =====================================================================
# 3. 2つのネットワークの定義と学習
# =====================================================================

# ① 普通のNN (ReLUを使った、少し深くて重いモデル)
class HeavyReLU_NN(nn.Module):
    def __init__(self):
        super(HeavyReLU_NN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x): return self.net(x)

# ② 提案手法の有理NN (層を減らしたスリムなモデル)
class SlimRational_NN(nn.Module):
    def __init__(self):
        super(SlimRational_NN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            RationalActivation(), # 自作した有理関数を活性化関数に指定！
            nn.Linear(32, 1)
        )
    def forward(self, x): return self.net(x)

# それぞれのパラメータ数をカウント
model_relu = HeavyReLU_NN()
model_rational = SlimRational_NN()
params_relu = sum(p.numel() for p in model_relu.parameters())
params_rational = sum(p.numel() for p in model_rational.parameters())

print(f"普通のReLUモデルの総パラメータ数: {params_relu}")
print(f"提案の有理関数モデルの総パラメータ数: {params_rational} (約 {params_rational/params_relu*100:.1f}% に軽量化！)")

# --- 学習の実行 ---
def train_model(model, epochs=3001, lr=0.01):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        optimizer.zero_grad()
        loss = criterion(model(X_train_tensor), Z_train_tensor)
        loss.backward()
        optimizer.step()
    return model

print("\n普通のReLUモデルを学習中...")
model_relu = train_model(model_relu)
print("提案の有理関数モデルを学習中...")
model_rational = train_model(model_rational)


# =====================================================================
# 4. 評価と2D断面図による比較
# =====================================================================
x_grid = np.linspace(0, 4, 60)
y_grid = np.linspace(1, 5, 60)
X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
X_eval_flat = np.column_stack([X_mesh.ravel(), Y_mesh.ravel()])
X_test_tensor = torch.tensor(X_eval_flat, dtype=torch.float32)

Z_true = complex_test_function(X_mesh, Y_mesh)

with torch.no_grad():
    Z_pred_relu = model_relu(X_test_tensor).numpy().reshape(X_mesh.shape)
    Z_pred_rational = model_rational(X_test_tensor).numpy().reshape(X_mesh.shape)

# y = 1.5 の直線に沿ったトゲのふもと断面データを抽出
y_target = 1.5
y_idx = (np.abs(y_grid - y_target)).argmin()

z_true_line = Z_true[y_idx, :]
z_relu_line = Z_pred_relu[y_idx, :]
z_rational_line = Z_pred_rational[y_idx, :]

# Plotlyグラフの描画
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_grid, y=z_true_line, mode='lines', name='True Function (正解)', line=dict(color='black', width=2, dash='dash')))
fig.add_trace(go.Scatter(x=x_grid, y=z_relu_line, mode='lines', name=f'Heavy ReLU NN (パラメータ数: {params_relu})', line=dict(color='red', width=2)))
fig.add_trace(go.Scatter(x=x_grid, y=z_rational_line, mode='lines', name=f'Slim Rational NN (パラメータ数: {params_rational})', line=dict(color='green', width=3)))

fig.update_layout(
    title='【検証結果】活性化関数を有理関数にすることによる軽量化と高精度化のトレードオフ突破',
    xaxis_title='x 座標', yaxis_title='z 軸の高さ',
    xaxis_range=[0.5, 2.0], yaxis_range=[-1.0, 15.0], width=1000, height=500
)
fig.show()

普通のReLUモデルの総パラメータ数: 33537
提案の有理関数モデルの総パラメータ数: 136 (約 0.4% に軽量化！)

普通のReLUモデルを学習中...
提案の有理関数モデルを学習中...


In [7]:
fig.update_layout(
    title='【検証結果】活性化関数を有理関数にすることによる軽量化と高精度化のトレードオフ突破',
    xaxis_title='x 座標', yaxis_title='z 軸の高さ',
    xaxis_range=[0.0, 5.0], yaxis_range=[-20.0, 200.0], width=1000, height=500
)
fig.show()